In [1]:
import os
import sys
import time
import shutil
import math

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

import torchvision
import torchvision.transforms as transforms

# Google Colab: mount Google Drive and work from the copy of the software/ folder there (it must contain models/),
# so the imports below work and checkpoints in result/ survive a runtime reset. Skipped when running locally.
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    colab_dir = '/content/drive/MyDrive/ece284fa26/software'   # change to where you uploaded the software/ folder
    os.chdir(colab_dir)
    sys.path.insert(0, colab_dir)

from models import *


global best_prec
if torch.cuda.is_available():
    device = torch.device('cuda')
    cudnn.benchmark = True       # fixed 32x32 input -> let cuDNN pick the fastest conv kernels
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('device:', device)

device: cuda


In [2]:
print('=> Building model...')

batch_size = 128
model_name = "VGG16_quant"
model = VGG16_quant()

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)

        input, target = input.to(device), target.to(device)

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))



def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):

            input, target = input.to(device), target.to(device)

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].reshape(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """QAT fine-tuning: the lr is divided by 10 at epochs 15 and 25"""
    adjust_list = [15, 25]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1

=> Building model...
Files already downloaded and verified
Files already downloaded and verified


In [3]:
fp_path = 'result/vgg16/model_best.pth.tar'
fp_checkpoint = torch.load(fp_path, map_location='cpu')
print('floating-point VGG16 (HW3): best_prec = {:.2f}%'.format(fp_checkpoint['best_prec']))

keys = model.load_state_dict(fp_checkpoint['state_dict'], strict=False)
print('loaded keys    :', len(fp_checkpoint['state_dict']) - len(keys.unexpected_keys))
print('unexpected keys:', keys.unexpected_keys)
print('missing keys   :', len(keys.missing_keys), '->', sorted({k.split('.')[-1] for k in keys.missing_keys}))

floating-point VGG16 (HW3): best_prec = 92.80%
loaded keys    : 80
unexpected keys: []
missing keys   : 39 -> ['act_alpha', 'weight_q', 'wgt_alpha']


In [4]:
fp_model = VGG16()
fp_model.load_state_dict(fp_checkpoint['state_dict'])
fp_model.to(device).eval()

fp_convs = [m for m in fp_model.features if isinstance(m, nn.Conv2d)]
q_convs  = [m for m in model.features if isinstance(m, QuantConv2d)]
calib_inputs = [[] for _ in fp_convs]

def save_calib_input(k):
    def hook(module, module_in):
        x = module_in[0].detach().abs().flatten()
        x = x[x > 0]                                    # skip the zeros produced by ReLU
        calib_inputs[k].append(x[torch.randint(x.numel(), (200_000,), device=x.device)].cpu())  # random sample
    return hook

handles = [conv.register_forward_pre_hook(save_calib_input(k)) for k, conv in enumerate(fp_convs)]
with torch.no_grad():
    for b, (images, _) in enumerate(trainloader):
        fp_model(images.to(device))
        if b == 3:                                      # 4 batches = 512 training images
            break
for h in handles:
    h.remove()

for conv, xs in zip(q_convs, calib_inputs):
    conv.act_alpha.data.fill_(torch.quantile(torch.cat(xs), 0.999).item())
print('act_alpha per QuantConv2d:', [round(conv.act_alpha.item(), 2) for conv in q_convs])
del fp_model, calib_inputs

act_alpha per QuantConv2d: [2.13, 3.82, 3.58, 1.91, 2.52, 2.17, 2.17, 2.91, 2.66, 3.09, 3.82, 3.04, 2.96]


In [5]:
run_training = False   # set to False to skip training and reuse the saved checkpoint in result/VGG16_quant

lr = 1e-3
weight_decay = 1e-4
epochs = 50
best_prec = 0

model.to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)

fdir = os.path.join('result', model_name)
if not os.path.exists(fdir):
    os.makedirs(fdir)


if run_training:
    for epoch in range(0, epochs):
        adjust_learning_rate(optimizer, epoch)

        train(trainloader, model, criterion, optimizer, epoch)

        # evaluate on test set
        print("Validation starts")
        prec = validate(testloader, model, criterion)

        # remember best precision and save checkpoint
        is_best = prec > best_prec
        best_prec = max(prec,best_prec)
        print('best acc: {:1f}'.format(best_prec))
        save_checkpoint({
            'epoch': epoch + 1,
            'state_dict': model.state_dict(),
            'best_prec': best_prec,
            'optimizer': optimizer.state_dict(),
        }, is_best, fdir)

In [6]:
PATH = os.path.join(fdir, 'model_best.pth.tar')
checkpoint = torch.load(PATH, map_location=device)
model.load_state_dict(checkpoint['state_dict'])
model.to(device)
model.eval()

correct = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # move to device
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()

print('checkpoint: {} (epoch {})'.format(PATH, checkpoint['epoch']))
print('\nTest set: Accuracy: {}/{} ({:.2f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))

checkpoint: result/VGG16_quant/model_best.pth.tar (epoch 29)

Test set: Accuracy: 9237/10000 (92.37%)



In [7]:
print(model.features)

layer = model.features[3]
print('\nmodel.features[3]:')
print(layer)
assert isinstance(layer, QuantConv2d) and layer.in_channels == 64 and layer.out_channels == 64

Sequential(
  (0): QuantConv2d(
    3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
    (weight_quant): weight_quantize_fn()
  )
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): QuantConv2d(
    64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
    (weight_quant): weight_quantize_fn()
  )
  (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (5): ReLU(inplace=True)
  (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (7): QuantConv2d(
    64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
    (weight_quant): weight_quantize_fn()
  )
  (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (9): ReLU(inplace=True)
  (10): QuantConv2d(
    128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
    (weight_quant): weight_quantize_fn()
  )
  (11):

In [8]:
class SaveInOut:
    def __init__(self):
        self.input = None
        self.output = None
    def __call__(self, module, module_in, module_out):
        self.input = module_in[0].detach().clone()    # clone: the ReLU before this layer is in-place
        self.output = module_out.detach().clone()

save = SaveInOut()
handle = layer.register_forward_hook(save)

images, labels = next(iter(testloader))
image = images[0:1].to(device)                        # one test image: [1, 3, 32, 32]
model.eval()
with torch.no_grad():
    model(image)
handle.remove()

x = save.input                                        # floating-point input of the layer
print('layer input :', tuple(x.shape))
print('layer output:', tuple(save.output.shape))

layer input : (1, 64, 32, 32)
layer output: (1, 64, 32, 32)


In [9]:
w_bit = 4
w_alpha = layer.weight_quant.wgt_alpha.detach()       # weight clipping value learned during training
w_delta = w_alpha / (2 ** (w_bit - 1) - 1)            # signed weights: integer levels -7..7
weight_q = layer.weight_q.detach()                    # quantized weight, refreshed by the hook pass above
w_int = (weight_q / w_delta).round()

print('w_alpha = {:.4f}, w_delta = {:.4f}'.format(w_alpha.item(), w_delta.item()))
print('max distance to nearest integer before rounding:', (weight_q / w_delta - w_int).abs().max().item())
print('w_int values:', w_int.unique().int().tolist())
print('w_int shape :', tuple(w_int.shape), ' [C_out, C_in, k, k]')

w_alpha = 2.9869, w_delta = 0.4267
max distance to nearest integer before rounding: 4.76837158203125e-07
w_int values: [-7, -6, -5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5, 6, 7]
w_int shape : (64, 64, 3, 3)  [C_out, C_in, k, k]


In [10]:
x_bit = 4
x_alpha = layer.act_alpha.detach()                    # activation clipping value learned during training
x_delta = x_alpha / (2 ** x_bit - 1)                  # unsigned activations: integer levels 0..15

with torch.no_grad():
    x_q = act_quantization(x_bit)(x, x_alpha)         # same activation quantizer the layer uses
x_int_batch = (x_q / x_delta).round()
x_int = x_int_batch[0]                                # one image: [C_in, H, W]

print('x_alpha = {:.4f}, x_delta = {:.4f}'.format(x_alpha.item(), x_delta.item()))
print('max distance to nearest integer before rounding:', (x_q / x_delta - x_int_batch).abs().max().item())
print('x_int values:', x_int.unique().int().tolist())
print('x_int shape :', tuple(x_int.shape), ' [C_in, H, W]')

x_alpha = 3.8889, x_delta = 0.2593
max distance to nearest integer before rounding: 9.5367431640625e-07
x_int values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
x_int shape : (64, 32, 32)  [C_in, H, W]


In [11]:
# Sanity check (as in HW5): conv(x_int, w_int) * x_delta * w_delta reproduces the layer's own output
with torch.no_grad():
    output_recovered = F.conv2d(x_int_batch, w_int, stride=1, padding=1) * x_delta * w_delta
print('max |recovered - layer output| = {:.3e}'.format((output_recovered - save.output).abs().max().item()))

# The tiling loops below run many small tensor operations, so keep them on the CPU (FAQ 12)
x_int = x_int.cpu()
w_int = w_int.cpu()
x_delta, w_delta = x_delta.item(), w_delta.item()

max |recovered - layer output| = 6.104e-05


## Tiling

In [12]:
array_size = 16                               # 16x16 weight-stationary systolic array
C_out, C_in, k, _ = w_int.shape               # 64, 64, 3
n_ic_tile = C_in // array_size                # 4 ic tiles (rows)
n_oc_tile = C_out // array_size               # 4 oc tiles (columns)
n_kij = k * k                                 # 9 kernel pixels, kij = ky*3 + kx

w_kij = w_int.reshape(C_out, C_in, n_kij)     # [oc, ic, kij]

w_tile = torch.zeros(n_ic_tile, n_oc_tile, n_kij, array_size, array_size)
for ic_tile in range(n_ic_tile):
    for oc_tile in range(n_oc_tile):
        for kij in range(n_kij):
            block = w_kij[oc_tile*array_size:(oc_tile+1)*array_size,     # 16 output channels
                          ic_tile*array_size:(ic_tile+1)*array_size,     # 16 input channels
                          kij]                                           # [16 (oc), 16 (ic)]
            w_tile[ic_tile, oc_tile, kij] = block.T                      # [16 rows (ic), 16 cols (oc)]

print('tiles:', n_ic_tile, 'ic tiles x', n_oc_tile, 'oc tiles =', n_ic_tile * n_oc_tile)
print('w_tile shape:', tuple(w_tile.shape), ' [ic_tile, oc_tile, kij, row (ic), col (oc)]')

# Check every PE: the PE at (row r, col c) of tile (ic_tile, oc_tile) holds w_int[16*oc_tile + c, 16*ic_tile + r, ky, kx]
n_checked, n_mismatch = 0, 0
for ic_tile in range(n_ic_tile):
    for oc_tile in range(n_oc_tile):
        for kij in range(n_kij):
            ky, kx = kij // k, kij % k
            for r in range(array_size):
                for c in range(array_size):
                    n_checked += 1
                    n_mismatch += int(w_tile[ic_tile, oc_tile, kij, r, c] !=
                                      w_int[array_size*oc_tile + c, array_size*ic_tile + r, ky, kx])
print('PE weights checked: {}, mismatches: {}'.format(n_checked, n_mismatch))

tiles: 4 ic tiles x 4 oc tiles = 16
w_tile shape: (4, 4, 9, 16, 16)  [ic_tile, oc_tile, kij, row (ic), col (oc)]
PE weights checked: 36864, mismatches: 0


In [13]:
padding = 1
stride = 1
_, H, W = x_int.shape                                  # 32, 32
H_pad, W_pad = H + 2*padding, W + 2*padding            # 34, 34

a_pad = torch.zeros(C_in, H_pad, W_pad)                # zero padding in the integer domain
a_pad[:, padding:padding+H, padding:padding+W] = x_int
a_pad = a_pad.reshape(C_in, -1)                        # merge y and x into nij = y*34 + x
n_nij = a_pad.size(1)                                  # 34*34 = 1156 time steps

a_tile = torch.zeros(n_ic_tile, n_oc_tile, array_size, n_nij)
for ic_tile in range(n_ic_tile):
    for oc_tile in range(n_oc_tile):                   # same activations for every oc tile
        a_tile[ic_tile, oc_tile] = a_pad[ic_tile*array_size:(ic_tile+1)*array_size, :]

print('a_pad shape :', tuple(a_pad.shape), ' [C_in, nij]')
print('a_tile shape:', tuple(a_tile.shape), ' [ic_tile, oc_tile, ic index (row #), nij (time step)]')

a_pad shape : (64, 1156)  [C_in, nij]
a_tile shape: (4, 4, 16, 1156)  [ic_tile, oc_tile, ic index (row #), nij (time step)]


## Partial Sums

In [14]:
psum = torch.zeros(n_ic_tile, n_oc_tile, array_size, n_nij, n_kij)
for ic_tile in range(n_ic_tile):
    for oc_tile in range(n_oc_tile):
        for kij in range(n_kij):
            w_blk = w_tile[ic_tile, oc_tile, kij]               # [16 rows (ic), 16 cols (oc)]
            a_blk = a_tile[ic_tile, oc_tile]                    # [16 rows (ic), 1156 time steps]
            psum[ic_tile, oc_tile, :, :, kij] = w_blk.T @ a_blk  # [16 cols (oc), 1156 time steps]

print('psum shape:', tuple(psum.shape), ' [ic_tile, oc_tile, oc index (col #), nij (time step), kij]')

psum shape: (4, 4, 16, 1156, 9)  [ic_tile, oc_tile, oc index (col #), nij (time step), kij]


In [15]:
# Cross-check one tile and kernel pixel one time step at a time, as the array computes it:
# the psum of each column flows north -> south, and the PE at row r adds w[r, c] * a[r] to it.
ic_tile, oc_tile, kij = 1, 2, 4
w_blk = w_tile[ic_tile, oc_tile, kij]
psum_step = torch.zeros(array_size, n_nij)
for nij in range(n_nij):                                 # time step
    a_in = a_tile[ic_tile, oc_tile, :, nij]              # 16 activations entering the 16 rows
    col_psum = torch.zeros(array_size)                   # psum entering the top of each column
    for r in range(array_size):                          # north -> south through the rows
        col_psum = col_psum + w_blk[r, :] * a_in[r]
    psum_step[:, nij] = col_psum

print('time-step simulation vs. matrix product, max abs difference:',
      (psum_step - psum[ic_tile, oc_tile, :, :, kij]).abs().max().item())

time-step simulation vs. matrix product, max abs difference: 0.0


In [16]:
o_H = (H_pad - k) // stride + 1                      # 32
o_W = (W_pad - k) // stride + 1                      # 32
n_onij = o_H * o_W                                   # 1024 output pixels

out = torch.zeros(C_out, n_onij)
for o_nij in range(n_onij):
    oy, ox = o_nij // o_W, o_nij % o_W
    for kij in range(n_kij):
        ky, kx = kij // k, kij % k
        nij = (oy*stride + ky) * W_pad + (ox*stride + kx)   # padded input pixel used by (oy, ox) at (ky, kx)
        for oc_tile in range(n_oc_tile):
            out[oc_tile*array_size:(oc_tile+1)*array_size, o_nij] += \
                psum[:, oc_tile, :, nij, kij].sum(dim=0)    # accumulate over the 4 ic tiles

print('out shape:', tuple(out.shape), ' [oc index, o_nij]')

out shape: (64, 1024)  [oc index, o_nij]


In [17]:
conv_ref = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, bias=False)
conv_ref.weight = nn.Parameter(w_int)
with torch.no_grad():
    out_ref = conv_ref(x_int.unsqueeze(0)).reshape(C_out, -1)   # [64, 1024]

abs_diff = (out - out_ref).abs()

print('a_tile  shape:', tuple(a_tile.shape))
print('psum    shape:', tuple(psum.shape))
print('out     shape:', tuple(out.shape))
print('out_ref shape:', tuple(out_ref.shape))
print()
print('sum of |out - out_ref| =', abs_diff.sum().item())
print('max of |out - out_ref| =', abs_diff.max().item())

a_tile  shape: (4, 4, 16, 1156)
psum    shape: (4, 4, 16, 1156, 9)
out     shape: (64, 1024)
out_ref shape: (64, 1024)

sum of |out - out_ref| = 0.0
max of |out - out_ref| = 0.0


In [18]:
# For reference: rescaling the mapped integer output by the step sizes recovers the layer's floating-point output
out_recovered = out * x_delta * w_delta
layer_out = save.output[0].reshape(C_out, -1).cpu()
print('max |out * x_delta * w_delta - layer output| = {:.3e}'.format((out_recovered - layer_out).abs().max().item()))

max |out * x_delta * w_delta - layer output| = 6.104e-05
